# 지식 주입 Fine-tuning — 가상 쇼핑몰 '무드포트'
**모델:** Qwen2.5-1.5B-Instruct  
**방법:** QLoRA (HuggingFace + bitsandbytes)  
**환경:** Google Colab (무료 티어, T4 GPU)

---

### 이 노트북에서 배우는 것
앞서 본 *조선시대 말투* 노트북은 **문체(스타일)** 를 학습시켰습니다.  
이번엔 한 단계 더 들어가서, 모델이 **원래 모르던 사실(지식)** 을 fine-tuning으로 주입합니다.

가상의 향·디퓨저 쇼핑몰 **무드포트**의 사내 정보(배송비·환불 규정·멤버십·마스코트 등)를  
모델 가중치에 직접 새겨 넣고, 학습 후 **세 종류의 질문**으로 찔러봅니다.

> 💡 핵심 메시지: fine-tuning으로 지식 주입은 **됩니다.** 다만 *제대로 하는 법*과 *한계*가 분명히 있습니다.  
> 이 노트북은 그 둘을 **눈으로** 보게 만드는 게 목적입니다.

이 노트북은 Unsloth를 사용하지 않고, **순수 HuggingFace Transformers + bitsandbytes + PEFT**만으로 QLoRA 학습을 수행합니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [1]:
!pip install -q --upgrade transformers datasets accelerate peft trl bitsandbytes evaluate rouge-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.

In [2]:
# 현재 호환되지 않는 최신 패키지들을 다운그레이드하고 가속화 라이브러리를 재설치합니다.
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q --upgrade accelerate transformers trl datasets

지저분한 경고 메세지를 띄우지 않기 위한 코드

In [3]:
import warnings
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

## 2. 모델 로드
HuggingFace Transformers + bitsandbytes로 4bit 양자화된 모델을 불러옵니다.

### BitsAndBytesConfig 설명

| 파라미터 | 값 | 설명 |
|---------|---|------|
| `load_in_4bit` | `True` | 모델 가중치를 4bit로 양자화하여 로드 (메모리 약 4배 절약) |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 양자화. 정규분포를 따르는 가중치에 최적화된 데이터 타입 |
| `bnb_4bit_compute_dtype` | `float16` | 양자화된 가중치를 연산할 때 사용하는 정밀도. float16이 T4에서 가장 빠름 |
| `bnb_4bit_use_double_quant` | `True` | 양자화 상수를 한 번 더 양자화하여 추가 메모리 절약 (약 0.4GB) |

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda:0",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ 모델 로드 완료")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ 모델 로드 완료


## 2.5 학습 *전* 베이스 모델에게 먼저 물어보기
학습을 시작하기 전에, 아직 아무것도 안 배운 베이스 모델이 무드포트를 아는지 확인합니다.  
무드포트는 **가상 회사**라 모델이 알 리가 없습니다. 그런데도 모델이 어떻게 반응하는지 보세요.

> 이 셀에서 에러가 나면 건너뛰어도 됩니다 — 학습 후 결과(6번)만 봐도 핵심은 전달됩니다.

In [5]:
def generate(prompt, model, max_new_tokens=120):
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=False,
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

before_questions = [
    "무드포트 무료배송 기준이 얼마예요?",
    "무드포트 본사는 어디에 있어요?",
]

for q in before_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

Q: 무드포트 무료배송 기준이 얼마예요?
A: 죄송합니다, 저는 실시간 웹사이트를 확인하거나 특정 웹사이트의 정보를 제공할 수 없습니다. 무드포트의 배송 가격이나 기준은 해당 웹사이트나 애플리케이션에 직접적으로 확인해야 합니다. 이는 개인적인 상황과 상품의 특성에 따라 달라질 수 있으므로, 구매 전에 최신 정보를 확인하는 것이 좋습니다.
--------------------------------------------------
Q: 무드포트 본사는 어디에 있어요?
A: 죄송합니다, 저는 실시간 정보를 제공할 수 없습니다. 현재 위치에 대한 자세한 정보는 해당 기업의 공식 웹사이트나 뉴스 사이트에서 확인하실 수 있습니다.
--------------------------------------------------


👉 베이스 모델은 무드포트를 모르니까, 보통 **얼버무리거나 그럴듯하게 지어냅니다.**  
이게 우리가 학습으로 바꿔줄 출발점입니다.

## 3. LoRA 설정
학습할 레이어만 골라 메모리를 아낍니다. *문체* 학습 때와 한 가지가 다릅니다 → **`r` 값을 키웁니다.**

#### `r = 32` (LoRA rank) — 지식 주입은 더 큰 용량이 필요
`r`은 LoRA가 학습하는 작은 행렬의 크기입니다. 값이 클수록 모델에 **더 많은 정보를 담을 공간**이 생깁니다.

| r 값 | 용도 |
|------|------|
| 8 | 가벼운 스타일 변환 |
| 16 | 가장 흔한 선택 (조선시대 말투 노트북에서 사용) |
| **32~64** | **여러 개의 사실을 외워야 하는 지식 주입** |

말투는 하나의 *패턴*만 익히면 되지만, 지식은 *서로 다른 사실 여러 개*를 따로따로 기억해야 하므로 공간이 더 필요합니다.

#### `lora_alpha = 32`
학습한 변화량을 얼마나 세게 반영할지 정합니다. 보통 `r`과 같거나 2배로 둡니다. 여기서는 `r`과 동일하게 맞췄습니다.

#### `lora_dropout = 0.05`
순수 HuggingFace 버전에서는 Unsloth의 최적화 커널이 없으므로, 소량의 dropout(0.05)을 적용하여 과적합을 방지합니다.

#### `target_modules`
Attention 레이어(q/k/v/o_proj)뿐 아니라 **MLP 레이어(gate/up/down_proj)** 까지 학습 대상에 넣습니다.  
연구상 모델이 *사실을 저장하는 곳*이 주로 MLP 레이어라, 지식 주입에서는 이 부분을 건드리는 게 특히 중요합니다.

나머지(`bias="none"`, gradient checkpointing)는 조선시대 노트북과 동일합니다.

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()

model.print_trainable_parameters()
print("✅ LoRA 설정 완료")

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364
✅ LoRA 설정 완료


## 4. 데이터 준비 — *여기가 이 노트북의 핵심*

### ❌ 흔한 실수: 사실 하나당 질문 하나
```
Q: 무료배송 기준이 얼마예요?  →  A: 4만원 이상이요
```
이렇게 **한 방향으로 한 번만** 가르치면, 모델은 사실이 아니라 *그 문장의 겉모습*을 외웁니다.  
그래서 질문을 조금만 바꾸면 무너지고, 거꾸로 물으면("4만원은 무슨 기준?") 답을 못 합니다.  
이걸 **Reversal Curse(역방향 저주)** 라고 부릅니다.

### ✅ 제대로 하는 법: 한 사실을 여러 각도로 부풀리기 (augmentation)
같은 사실 하나를 **여러 표현·여러 방향·여러 맥락**으로 펼쳐서 가르칩니다.

| 패턴 | 예시 |
|------|------|
| 정방향 질문 | "무료배송 기준이 얼마예요?" → "4만원 이상이요" |
| 역방향 질문 | "4만원은 무슨 기준이에요?" → "무료배송 최소 금액이요" |
| 경계/반대 | "3만 8천원이면 배송비 내요?" → "네, 4만원 미만이라 붙어요" |
| 평서문 | "무드포트는 4만원 이상 구매 시 무료배송을 제공합니다." |
| 다른 표현 | "배송비 안 내려면 얼마 채워요?" → "4만원까지요" |

아래에서는 이런 식으로 부풀려 놓은 `moodport_qa.json` 파일을 불러와 학습 데이터로 사용합니다.  
하나의 사실 블록 안에 여러 표현이 들어 있는 구조를 직접 확인해 보세요.

In [7]:
# 파일 업로드
import json

# 보통 '/kaggle/input/[지정한데이터셋이름]/moodport_qa.json' 형식입니다.
file_path = "/kaggle/input/datasets/amyhong0/moodport/moodport_qa.json"

with open(file_path, "r", encoding="utf-8") as f:
    knowledge = json.load(f)

# 사실 블록을 학습용 (질문, 답) 쌍으로 펼치기
raw = []
for item in knowledge:
    for qa in item["qa"]:
        raw.append({"instruction": qa["question"], "output": qa["answer"]})

print(f"✅ 파일을 성공적으로 불러왔습니다! 총 {len(raw)}개의 QA 쌍이 준비되었습니다.")

✅ 파일을 성공적으로 불러왔습니다! 총 60개의 QA 쌍이 준비되었습니다.


In [8]:
import json

with open(file_path, "r", encoding="utf-8") as f:
    knowledge = json.load(f)

# 사실 블록을 학습용 (질문, 답) 쌍으로 펼치기
raw = []
for item in knowledge:
    for qa in item["qa"]:
        raw.append({"instruction": qa["question"], "output": qa["answer"]})

In [9]:
import json
from datasets import Dataset

def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw).map(format_example)

# 학습/평가 데이터셋 분리 (90:10)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ 데이터셋 준비 완료: 학습 {len(train_dataset)}개 / 평가 {len(eval_dataset)}개")
print("\n--- 샘플 확인 ---")
print(train_dataset[0]["text"])

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

✅ 데이터셋 준비 완료: 학습 54개 / 평가 6개

--- 샘플 확인 ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
무드포트 회원 등급 종류 알려줘.<|im_end|>
<|im_start|>assistant
낮은 순서로 데일리 → 무드 → 시그니처 등급이 있어요.<|im_end|>



## 5. 학습 실행
약 5~10분 소요됩니다.

#### 조선시대 노트북과 다른 점: `num_train_epochs`를 키웁니다
문체는 3 epoch이면 충분했지만, **사실을 확실히 외우게 하려면 같은 데이터를 더 여러 번** 보여줘야 합니다.  
여기서는 **8 epoch**으로 설정했습니다.

> ⚖️ 단, epoch을 더 올리면 **외우긴 더 잘 외우지만**, 안 가르친 질문에는 *더 자신있게 거짓말*하게 됩니다.  
> 이게 바로 6번에서 확인할 **과적합의 두 얼굴**입니다.

#### 평가 지표: ROUGE & BLEU
매 epoch마다 평가 데이터셋에 대해 **ROUGE**(정답과 겹치는 단어 비율)와 **BLEU**(n-gram 정밀도)를 측정합니다.  
학습이 진행될수록 점수가 오르는 것을 확인할 수 있습니다.

> 여기서 측정하는 점수는 **teacher forcing** 방식(정답 토큰을 보면서 다음 토큰을 예측)이라,  
> 실제 자유 생성 결과와는 차이가 있을 수 있습니다. 학습 추이를 관찰하는 용도로 활용하세요.

In [10]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import torch
import evaluate

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

# 매 epoch 끝에 평가셋을 실제로 generate해서 ROUGE/BLEU를 측정하는 콜백
# (이 transformers 버전은 eval 시 logits를 넘겨주지 않아 compute_metrics로는 불가)
class GenMetricsCallback(TrainerCallback):
    def __init__(self, eval_dataset, tokenizer, max_new_tokens=128):
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        model.eval()
        preds, refs = [], []
        for ex in self.eval_dataset:
            messages = [{"role": "user", "content": ex["instruction"]}]
            inputs = self.tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_tensors="pt", return_dict=False,
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    input_ids=inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                )
            text = self.tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
            preds.append(text if text.strip() else " ")
            refs.append(ex["output"])

        rouge = rouge_metric.compute(predictions=preds, references=refs)
        bleu  = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
        print(f"\n📊 [Epoch {state.epoch:.0f}] rougeL={rouge['rougeL']:.4f}  bleu={bleu['bleu']:.4f}")
        model.train()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[GenMetricsCallback(eval_dataset, tokenizer)],
    args=SFTConfig(
        max_length=512,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=8,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=False,
        bf16=False,
        logging_steps=10,
        eval_strategy="epoch",
        output_dir="./moodport_output",
        report_to="none",
        gradient_checkpointing_kwargs={"use_reentrant": False},        
    ),
)

trainer.train()
print("✅ 학습 완료")

Adding EOS to train dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]


📊 [Epoch 1] rougeL=0.0000  bleu=0.0000
{'eval_loss': '5.875', 'eval_runtime': '0.43', 'eval_samples_per_second': '13.95', 'eval_steps_per_second': '2.326', 'eval_entropy': '2.331', 'eval_num_tokens': '3599', 'eval_mean_token_accuracy': '0.5398', 'epoch': '1'}

📊 [Epoch 2] rougeL=0.0000  bleu=0.0000
{'eval_loss': '2.901', 'eval_runtime': '0.4047', 'eval_samples_per_second': '14.83', 'eval_steps_per_second': '2.471', 'eval_entropy': '1.545', 'eval_num_tokens': '7198', 'eval_mean_token_accuracy': '0.7198', 'epoch': '2'}
{'loss': '5.614', 'grad_norm': '3.156', 'learning_rate': '0.0001847', 'entropy': '1.992', 'num_tokens': '9318', 'mean_token_accuracy': '0.5624', 'epoch': '2.571'}

📊 [Epoch 3] rougeL=0.0000  bleu=0.0000
{'eval_loss': '2.071', 'eval_runtime': '0.4161', 'eval_samples_per_second': '14.42', 'eval_steps_per_second': '2.403', 'eval_entropy': '0.9709', 'eval_num_tokens': '1.08e+04', 'eval_mean_token_accuracy': '0.7841', 'epoch': '3'}

📊 [Epoch 4] rougeL=0.1667  bleu=0.0000
{'eva

## 6. 결과 — 빛과 그림자 직접 확인하기
학습한 모델에게 **세 종류**의 질문을 던집니다. 버킷마다 결과가 어떻게 갈리는지 보세요.

- **① 가르친 그대로** — 정방향 질문. → 잘 맞혀야 정상 ✅
- **② 비틀기** — 역방향·경계 질문. *augmentation을 했기 때문에* 대체로 버팁니다 💪
- **③ 안 가르친 것** — 학습에 없던 사실. → **모른다고 안 하고 자신있게 지어냅니다** ⚠️

③번이 핵심입니다. "분포 안은 외우지만, 한 발만 벗어나면 침묵이 아니라 **그럴듯한 환각**" — fine-tuning 지식 주입의 본질적 한계입니다.

In [11]:
buckets = {
    "① 가르친 그대로 (정방향)": [
        "무드포트 무료배송 기준이 얼마예요?",
        "무드포트 본사는 어디에 있어요?",
        "무드포트 환불은 언제까지 가능해요?",
        "무드포트 마스코트가 뭐예요?",
    ],
    "② 비틀기 (역방향·경계)": [
        "4만원은 무드포트에서 무슨 기준 금액이에요?",   # 역방향
        "3만 5천원어치 담으면 배송비 붙어요?",          # 경계
        "포포가 누구예요?",                              # 다른 각도
        "받은 지 20일 됐는데 환불돼요?",                # 경계(거절)
    ],
    "③ 안 가르친 것 (환각 주의)": [
        "무드포트 창립자 이름이 뭐예요?",
        "무드포트 직원 수는 몇 명이에요?",
        "무드포트 작년 매출이 얼마예요?",
    ],
}

for title, questions in buckets.items():
    print("=" * 60)
    print(title)
    print("=" * 60)
    for q in questions:
        print(f"Q: {q}")
        print(f"A: {generate(q, model)}")
        print("-" * 50)

① 가르친 그대로 (정방향)
Q: 무드포트 무료배송 기준이 얼마예요?
A: 1만원 이상 구매시 무료배송이에요.
--------------------------------------------------
Q: 무드포트 본사는 어디에 있어요?
A: 무드포트는 서울특별시에 본사가 있어요.
--------------------------------------------------
Q: 무드포트 환불은 언제까지 가능해요?
A: 14일 내에 환불 신청이 가능해요.
--------------------------------------------------
Q: 무드포트 마스코트가 뭐예요?
A: 무드포트의 마스코트는 '무드'예요.
--------------------------------------------------
② 비틀기 (역방향·경계)
Q: 4만원은 무드포트에서 무슨 기준 금액이에요?
A: 무드포트는 10만원을 기준으로 합니다.
--------------------------------------------------
Q: 3만 5천원어치 담으면 배송비 붙어요?
A: 아니요, 배송비 무료예요.
--------------------------------------------------
Q: 포포가 누구예요?
A: 무드레터를 출시한 배우이에요.
--------------------------------------------------
Q: 받은 지 20일 됐는데 환불돼요?
A: 아니요, 환불이 가능하지요.
--------------------------------------------------
③ 안 가르친 것 (환각 주의)
Q: 무드포트 창립자 이름이 뭐예요?
A: 무드포트의 창립자는 김지현 씨예요.
--------------------------------------------------
Q: 무드포트 직원 수는 몇 명이에요?
A: 100명 이하예요.
--------------------------------------------------
Q: 무드포트 작년 

### 🔎 결과를 이렇게 읽으세요
- ①이 잘 나오면 → **지식 주입은 성공했습니다.** fine-tuning으로 사실이 가중치에 새겨진 것.
- ②도 잘 나오면 → **augmentation 덕분.** 한 방향으로만 가르쳤다면 여기서 무너졌을 겁니다.
- ③에서 모델이 그럴듯한 이름·숫자를 지어내면 → **이게 핵심 교훈.**  
  모델은 "모릅니다"라고 말하는 법을 배우지 않았습니다. 가르친 범위를 벗어나면 *멈추는 게 아니라 채워 넣습니다.*

> 그래서 실무에서는 **자주 바뀌고 정확해야 하는 사실(가격·재고·규정)은 RAG로**,  
> **잘 안 바뀌는 행동·말투·응대 방식은 fine-tuning으로** 나눠서 씁니다.  
> "fine-tuning은 지식을 *못* 넣는다"가 아니라, **"넣을 수 있지만 한계를 알고 써야 한다"** 가 정확한 결론입니다.